# Relay: your own AI model on Colab

This notebook runs **Qwen 3.8 27B** on Colab's GPU and lets the Relay help page use it for answers.
While this notebook is running, the help page asks your model first. When the session ends,
the help page automatically goes back to Gemini and Cloudflare, so it never stops working.

**Before you start**
1. *Runtime → Change runtime type →* **L4** or **A100** GPU. The model needs about 20 GB of GPU memory;
   on a T4 (16 GB) it still runs, but slowly.
2. Click the **key icon** on the left (*Secrets*), add a secret named `RELAY_COLAB_KEY` with the same
   password as the help page's `COLAB_KEY`, and turn on *Notebook access*.
3. *Runtime → Run all.* The first run downloads the model (about 17 GB, a few minutes).

The address this notebook creates is temporary and protected by your password. Nothing here
stores chats: each question is answered and forgotten.

In [ ]:
# 1. Check the GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "No GPU: choose a GPU runtime first (Runtime -> Change runtime type)."

In [ ]:
# 2. Install Ollama and download the model
import subprocess, time, urllib.request
MODEL = "qwen3.8:27b"
# Ollama's installer needs zstd to unpack, which Colab doesn't include.
!apt-get -qq update > /dev/null && apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh 2>&1 | tail -3
import shutil
assert shutil.which("ollama"), "Ollama didn't install: read the lines above."
ollama = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, start_new_session=True)
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/version", timeout=2); break
    except Exception:
        time.sleep(1)
!ollama pull {MODEL}
!ollama run {MODEL} --think=false "Reply with just: ready" 2>/dev/null

In [ ]:
# 3. Start the password-protected model server
import os
from google.colab import userdata
os.environ["RELAY_COLAB_KEY"] = userdata.get("RELAY_COLAB_KEY")
os.environ["RELAY_MODEL"] = MODEL
assert len(os.environ["RELAY_COLAB_KEY"]) >= 24, "Use a long password (24+ characters) for RELAY_COLAB_KEY."
!pip -q install fastapi==0.115.6 uvicorn==0.34.0 httpx==0.28.1
!curl -fsSL -o relay_colab_server.py https://raw.githubusercontent.com/viaanshetty3-debug/relay-whatsapp-mcp/main/colab/relay_colab_server.py
server = subprocess.Popen(["python", "-m", "uvicorn", "relay_colab_server:app", "--port", "8000", "--log-level", "warning"], start_new_session=True)
time.sleep(4)
!curl -s -H "Authorization: Bearer $RELAY_COLAB_KEY" http://127.0.0.1:8000/health

In [ ]:
# 4. Open a temporary public address (Cloudflare quick tunnel: no account needed)
import re
!curl -fsSL -o cloudflared https://github.com/cloudflare/cloudflared/releases/download/2026.10.7/cloudflared-linux-amd64 && chmod +x cloudflared
tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8000"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, start_new_session=True)
ADDRESS = None
for line in tunnel.stdout:
    found = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if found:
        ADDRESS = found.group(0); break
import threading
threading.Thread(target=lambda: [None for _ in tunnel.stdout], daemon=True).start()  # keep its log from filling up
print("Your model's address:", ADDRESS)

In [ ]:
# 5. Tell the help page where your model is. Keep this cell running; stop it to switch back.
import json, urllib.request
HELP_PAGE = "https://relay-docs-ai.viaan-mcp.workers.dev"
def call(method, body=None):
    req = urllib.request.Request(HELP_PAGE + "/api/engine/colab", method=method,
        data=json.dumps(body).encode() if body is not None else None,
        headers={"Authorization": "Bearer " + os.environ["RELAY_COLAB_KEY"], "Content-Type": "application/json",
                 "User-Agent": "relay-colab-notebook/1"})  # Cloudflare blocks Python's default user agent
    return json.load(urllib.request.urlopen(req, timeout=15))
try:
    while True:
        alive = tunnel.poll() is None and server.poll() is None and ollama.poll() is None
        if not alive:
            print("Something stopped (model, server or tunnel). Run all cells again."); break
        try:
            call("POST", {"url": ADDRESS, "model": MODEL})
            print(time.strftime("%H:%M:%S"), "help page is using your model", end="\r")
        except Exception as e:
            print(time.strftime("%H:%M:%S"), "could not reach the help page:", e)
        time.sleep(120)
except KeyboardInterrupt:
    pass
finally:
    try: call("DELETE"); print("\nHelp page switched back to Gemini/Cloudflare.")
    except Exception: pass